In [3]:
import nba_api.stats.endpoints
import requests
import json
import pandas as pd
import nba_api

In [4]:
# Get Timberwolves player game logs for the season
wolves_player_logs = nba_api.stats.endpoints.PlayerGameLogs(season_nullable='2025-26',team_id_nullable='1610612750').get_data_frames()[0]

wolves_player_logs

,SEASON_YEAR,PLAYER_ID,PLAYER_NAME,NICKNAME,TEAM_ID,TEAM_ABBREVIATION,TEAM_NAME,GAME_ID,GAME_DATE,MATCHUP,...,PFD_RANK,PTS_RANK,PLUS_MINUS_RANK,NBA_FANTASY_PTS_RANK,DD2_RANK,TD3_RANK,WNBA_FANTASY_PTS_RANK,AVAILABLE_FLAG,MIN_SEC,TEAM_COUNT
0,2025-26,203497,Rudy Gobert,Rudy,1610612750,MIN,Minnesota Timberwolves,0022501020,2026-03-20T00:00:00,MIN vs. POR,...,164,157,284,58,1,4,103,1,38:24,1
1,2025-26,1630245,Ayo Dosunmu,Ayo,1610612750,MIN,Minnesota Timberwolves,0022501020,2026-03-20T00:00:00,MIN vs. POR,...,252,186,158,133,1,4,119,1,38:21,1
2,2025-26,1630183,Jaden McDaniels,Jaden,1610612750,MIN,Minnesota Timberwolves,0022501020,2026-03-20T00:00:00,MIN vs. POR,...,30,208,752,84,56,4,131,1,40:02,1
3,2025-26,203944,Julius Randle,Julius,1610612750,MIN,Minnesota Timberwolves,0022501020,2026-03-20T00:00:00,MIN vs. POR,...,64,135,446,256,56,4,223,1,36:44,1
4,2025-26,1628978,Donte DiVincenzo,Donte,1610612750,MIN,Minnesota Timberwolves,0022501020,2026-03-20T00:00:00,MIN vs. POR,...,252,274,799,330,56,4,266,1,35:08,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
859,2025-26,204060,Joe Ingles,Joe,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,588,613,373,652,56,4,626,1,16:10,1
860,2025-26,1642389,Zyon Pullin,Zyon,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,588,666,529,751,56,4,726,1,10:19,1
861,2025-26,1631262,Jules Bernard,Jules,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,588,746,557,770,56,4,776,1,4:10,1
862,2025-26,1641803,Tristen Newton,Tristen,1610612750,MIN,Minnesota Timberwolves,0012500028,2025-10-04T00:00:00,MIN @ DEN,...,588,666,373,790,56,4,776,1,3:08,1


In [5]:
# Filter for Rudy Gobert game logs
gobert_logs = wolves_player_logs[wolves_player_logs['PLAYER_NAME'] == 'Rudy Gobert'].copy()

print(f"Rudy Gobert games: {len(gobert_logs)}")
print(f"Total games in dataset: {len(gobert_logs)}")

# Display first few rows to verify
gobert_logs[['GAME_DATE', 'MATCHUP', 'WL', 'REB', 'PTS']].head(10)

Rudy Gobert games: 72
Total games in dataset: 72


,GAME_DATE,MATCHUP,WL,REB,PTS
0,2026-03-20T00:00:00,MIN vs. POR,L,15,18
9,2026-03-18T00:00:00,MIN vs. UTA,W,12,21
22,2026-03-17T00:00:00,MIN vs. PHX,W,19,9
34,2026-03-15T00:00:00,MIN @ OKC,L,7,2
43,2026-03-13T00:00:00,MIN @ GSW,W,9,18
54,2026-03-11T00:00:00,MIN @ LAC,L,5,9
69,2026-03-10T00:00:00,MIN @ LAL,L,12,3
80,2026-03-07T00:00:00,MIN vs. ORL,L,8,12
90,2026-03-05T00:00:00,MIN vs. TOR,W,12,18
108,2026-03-03T00:00:00,MIN vs. MEM,W,12,5


In [6]:
# Analyze Timberwolves record based on Rudy Gobert's rebound thresholds
from IPython.display import display

print("Timberwolves Record Based on Rudy Gobert's Rebounds")
print("=" * 80)

# Define rebound thresholds
thresholds = [
    ("Fewer than 10 rebounds", lambda x: x < 10),
    ("10 or more rebounds", lambda x: x >= 10),
    ("11 or more rebounds", lambda x: x >= 11),
    ("12 or more rebounds", lambda x: x >= 12),
    ("13 or more rebounds", lambda x: x >= 13),
    ("14 or more rebounds", lambda x: x >= 14),
    ("15 or more rebounds", lambda x: x >= 15)
]

results = []

for threshold_name, threshold_func in thresholds:
    # Filter games based on threshold
    filtered_games = gobert_logs[gobert_logs['REB'].apply(threshold_func)].copy()
    
    if len(filtered_games) > 0:
        wins = (filtered_games['WL'] == 'W').sum()
        losses = (filtered_games['WL'] == 'L').sum()
        total = wins + losses
        
        if total > 0:
            win_pct = wins / total
            avg_reb = filtered_games['REB'].mean()
            
            results.append({
                'Threshold': threshold_name,
                'Games': total,
                'Wins': wins,
                'Losses': losses,
                'Win %': f"{win_pct:.3f} ({win_pct*100:.1f}%)",
                'Avg REB': f"{avg_reb:.2f}"
            })

# Create results dataframe
results_df = pd.DataFrame(results)
display(results_df)

Timberwolves Record Based on Rudy Gobert's Rebounds


,Threshold,Games,Wins,Losses,Win %,Avg REB
0,Fewer than 10 rebounds,26,12,14,0.462 (46.2%),6.92
1,10 or more rebounds,46,31,15,0.674 (67.4%),13.54
2,11 or more rebounds,43,30,13,0.698 (69.8%),13.79
3,12 or more rebounds,39,29,10,0.744 (74.4%),14.08
4,13 or more rebounds,24,19,5,0.792 (79.2%),15.38
5,14 or more rebounds,19,16,3,0.842 (84.2%),16.00
6,15 or more rebounds,15,12,3,0.800 (80.0%),16.53


In [7]:
# Detailed breakdown for each threshold
print("Detailed Breakdown by Rebound Threshold")
print("=" * 100)

for threshold_name, threshold_func in thresholds:
    filtered_games = gobert_logs[gobert_logs['REB'].apply(threshold_func)].copy()
    
    if len(filtered_games) > 0:
        wins = (filtered_games['WL'] == 'W').sum()
        losses = (filtered_games['WL'] == 'L').sum()
        total = wins + losses
        
        if total > 0:
            win_pct = wins / total
            avg_reb = filtered_games['REB'].mean()
            
            print(f"\n{threshold_name}:")
            print(f"  Games: {total}")
            print(f"  Record: {wins}-{losses}")
            print(f"  Win Percentage: {win_pct:.3f} ({win_pct*100:.1f}%)")
            print(f"  Average Rebounds: {avg_reb:.2f}")
            
            # Show game details
            display_cols = ['GAME_DATE', 'MATCHUP', 'WL', 'REB', 'PTS', 'MIN']
            display_cols = [col for col in display_cols if col in filtered_games.columns]
            
            print(f"\n  Game Logs:")
            display(filtered_games[display_cols].sort_values('GAME_DATE', ascending=False))

Detailed Breakdown by Rebound Threshold

Fewer than 10 rebounds:
  Games: 26
  Record: 12-14
  Win Percentage: 0.462 (46.2%)
  Average Rebounds: 6.92

  Game Logs:


,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
34,2026-03-15T00:00:00,MIN @ OKC,L,7,2,28.533333
43,2026-03-13T00:00:00,MIN @ GSW,W,9,18,35.585000
54,2026-03-11T00:00:00,MIN @ LAC,L,5,9,26.133333
80,2026-03-07T00:00:00,MIN vs. ORL,L,8,12,33.116667
164,2026-02-11T00:00:00,MIN vs. POR,W,8,17,27.758333
178,2026-02-09T00:00:00,MIN vs. ATL,W,6,18,32.516667
186,2026-02-08T00:00:00,MIN vs. LAC,L,7,10,29.816667
252,2026-01-28T00:00:00,MIN @ DAL,W,6,6,20.600000
276,2026-01-25T00:00:00,MIN vs. GSW,L,5,4,24.100000
434,2025-12-27T00:00:00,MIN vs. BKN,L,8,6,31.433333



10 or more rebounds:
  Games: 46
  Record: 31-15
  Win Percentage: 0.674 (67.4%)
  Average Rebounds: 13.54

  Game Logs:


,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
0,2026-03-20T00:00:00,MIN vs. POR,L,15,18,38.400000
9,2026-03-18T00:00:00,MIN vs. UTA,W,12,21,28.766667
22,2026-03-17T00:00:00,MIN vs. PHX,W,19,9,36.841667
69,2026-03-10T00:00:00,MIN @ LAL,L,12,3,27.150000
90,2026-03-05T00:00:00,MIN vs. TOR,W,12,18,35.383333
108,2026-03-03T00:00:00,MIN vs. MEM,W,12,5,26.480000
114,2026-03-01T00:00:00,MIN @ DEN,W,15,7,37.568333
124,2026-02-26T00:00:00,MIN @ LAC,W,13,3,34.615000
132,2026-02-24T00:00:00,MIN @ POR,W,19,10,35.710000
153,2026-02-20T00:00:00,MIN vs. DAL,W,17,22,32.383333



11 or more rebounds:
  Games: 43
  Record: 30-13
  Win Percentage: 0.698 (69.8%)
  Average Rebounds: 13.79

  Game Logs:


...

,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
0,2026-03-20T00:00:00,MIN vs. POR,L,15,18,38.400000
22,2026-03-17T00:00:00,MIN vs. PHX,W,19,9,36.841667
114,2026-03-01T00:00:00,MIN @ DEN,W,15,7,37.568333
124,2026-02-26T00:00:00,MIN @ LAC,W,13,3,34.615000
132,2026-02-24T00:00:00,MIN @ POR,W,19,10,35.710000
153,2026-02-20T00:00:00,MIN vs. DAL,W,17,22,32.383333
198,2026-02-06T00:00:00,MIN vs. NOP,L,16,12,33.053333
228,2026-01-31T00:00:00,MIN @ MEM,W,16,9,30.800000
258,2026-01-26T00:00:00,MIN vs. GSW,W,17,15,34.983333
317,2026-01-16T00:00:00,MIN @ HOU,L,13,10,32.448333



14 or more rebounds:
  Games: 19
  Record: 16-3
  Win Percentage: 0.842 (84.2%)
  Average Rebounds: 16.00

  Game Logs:


,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
0,2026-03-20T00:00:00,MIN vs. POR,L,15,18,38.400000
22,2026-03-17T00:00:00,MIN vs. PHX,W,19,9,36.841667
114,2026-03-01T00:00:00,MIN @ DEN,W,15,7,37.568333
132,2026-02-24T00:00:00,MIN @ POR,W,19,10,35.710000
153,2026-02-20T00:00:00,MIN vs. DAL,W,17,22,32.383333
198,2026-02-06T00:00:00,MIN vs. NOP,L,16,12,33.053333
228,2026-01-31T00:00:00,MIN @ MEM,W,16,9,30.800000
258,2026-01-26T00:00:00,MIN vs. GSW,W,17,15,34.983333
340,2026-01-11T00:00:00,MIN vs. SAS,W,14,2,28.858333
367,2026-01-06T00:00:00,MIN vs. MIA,W,16,13,32.633333



15 or more rebounds:
  Games: 15
  Record: 12-3
  Win Percentage: 0.800 (80.0%)
  Average Rebounds: 16.53

  Game Logs:


,GAME_DATE,MATCHUP,WL,REB,PTS,MIN
0,2026-03-20T00:00:00,MIN vs. POR,L,15,18,38.400000
22,2026-03-17T00:00:00,MIN vs. PHX,W,19,9,36.841667
114,2026-03-01T00:00:00,MIN @ DEN,W,15,7,37.568333
132,2026-02-24T00:00:00,MIN @ POR,W,19,10,35.710000
153,2026-02-20T00:00:00,MIN vs. DAL,W,17,22,32.383333
198,2026-02-06T00:00:00,MIN vs. NOP,L,16,12,33.053333
228,2026-01-31T00:00:00,MIN @ MEM,W,16,9,30.800000
258,2026-01-26T00:00:00,MIN vs. GSW,W,17,15,34.983333
367,2026-01-06T00:00:00,MIN vs. MIA,W,16,13,32.633333
453,2025-12-23T00:00:00,MIN vs. NYK,W,16,11,37.750000


In [8]:
# Summary statistics
print("Rudy Gobert Rebounding Summary")
print("=" * 80)

if len(gobert_logs) > 0:
    print(f"\nTotal Games: {len(gobert_logs)}")
    print(f"Average Rebounds: {gobert_logs['REB'].mean():.2f}")
    print(f"Median Rebounds: {gobert_logs['REB'].median():.2f}")
    print(f"Min Rebounds: {gobert_logs['REB'].min()}")
    print(f"Max Rebounds: {gobert_logs['REB'].max()}")
    
    # Overall team record
    wins = (gobert_logs['WL'] == 'W').sum()
    losses = (gobert_logs['WL'] == 'L').sum()
    total = wins + losses
    
    if total > 0:
        win_pct = wins / total
        print(f"\nOverall Team Record: {wins}-{losses} ({win_pct:.3f} / {win_pct*100:.1f}%)")
    
    # Rebound distribution
    print(f"\nRebound Distribution:")
    rebound_counts = gobert_logs['REB'].value_counts().sort_index()
    print(rebound_counts)

Rudy Gobert Rebounding Summary

Total Games: 72
Average Rebounds: 11.15
Median Rebounds: 12.00
Min Rebounds: 3
Max Rebounds: 19

Overall Team Record: 43-29 (0.597 / 59.7%)

Rebound Distribution:
REB
3      1
4      1
5      3
6      5
7      4
8      8
9      4
10     3
11     4
12    15
13     5
14     4
15     4
16     5
17     2
18     2
19     2
Name: count, dtype: int64
